In [0]:
# Notebook: build_object_registry
# Structure Volume : /Volumes/{catalog}/{schema}/{schema}_volume/{schema}/fichier.sql

import os
import re
from pyspark.sql import Row

# ── Widgets ──────────────────────────────────────────────────────────────────
dbutils.widgets.text("catalog",         "dbe_dbx_internships")
dbutils.widgets.text("registry_schema", "log")
dbutils.widgets.text("volume_root",     "/Volumes/dbe_dbx_internships")
dbutils.widgets.text("schemas",         "dbo, datastore, datastore, datastore2, datastore3, datastore4, datastore5, datastore6, dwh, etl, ssas, staginghistoric, stagingintercompany")

catalog         = dbutils.widgets.get("catalog")
registry_schema = dbutils.widgets.get("registry_schema")
volume_root     = dbutils.widgets.get("volume_root")
schemas         = [s.strip() for s in dbutils.widgets.get("schemas").split(",") if s.strip()]

# ── Regex pour extraire le nom de l'objet créé ───────────────────────────────
OBJECT_PATTERN = re.compile(
    r"""
    CREATE\s+(?:OR\s+ALTER\s+)?
    (?P<obj_type>
        TABLE|VIEW|
        (?:INLINE\s+TABLE-VALUED\s+)?FUNCTION|
        (?:STORED\s+)?PROC(?:EDURE)?
    )\s+
    (?:\[?(?P<schema_prefix>\w+)\]?\.)?\s*
    \[?(?P<obj_name>\w+)\]?
    """,
    re.IGNORECASE | re.VERBOSE,
)

def detect_object_type(raw_type: str) -> str:
    t = raw_type.upper()
    if "VIEW"     in t: return "VIEW"
    if "FUNCTION" in t: return "FUNCTION"
    if "PROC"     in t: return "PROCEDURE"
    return "TABLE"

def read_sql_file(path: str) -> str:
    """Read a .sql file, trying UTF-16 (common for SSMS exports) then UTF-8."""
    # Check for BOM to pick the right encoding on the first try
    with open(path, "rb") as f:
        raw = f.read(4)

    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        encodings = ["utf-16"]
    else:
        encodings = ["utf-8-sig", "utf-8", "utf-16", "latin-1"]

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except (UnicodeDecodeError, UnicodeError):
            continue

    # Last resort: read as binary and decode lossily
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

# ── Scan ─────────────────────────────────────────────────────────────────────
rows = []

for schema in schemas:
    level3 = os.path.join(volume_root, schema, f"{schema}_volume", schema)

    if not os.path.isdir(level3):
        print(f"[SKIP] {level3} introuvable")
        continue

    sql_files = [f for f in sorted(os.listdir(level3)) if f.lower().endswith(".sql")]
    print(f"[{schema}] {len(sql_files)} fichiers .sql trouvés")

    for filename in sql_files:
        file_path = os.path.join(level3, filename)

        try:
            content = read_sql_file(file_path)
        except Exception as e:
            print(f"  [WARN] Impossible de lire {file_path}: {e}")
            continue

        match = OBJECT_PATTERN.search(content)
        if match:
            obj_name        = match.group("obj_name")
            obj_type        = detect_object_type(match.group("obj_type"))
            declared_schema = match.group("schema_prefix") or schema
        else:
            parts           = os.path.splitext(filename)[0].split(".")
            obj_name        = parts[1] if len(parts) >= 2 else parts[0]
            obj_type        = "UNKNOWN"
            declared_schema = schema

        rows.append(Row(
            catalog         = catalog,
            schema          = schema,
            declared_schema = declared_schema,
            object_name     = obj_name.lower(),
            object_type     = obj_type,
            file_path       = file_path,
            file_name       = filename,
        ))

print(f"\nTotal : {len(rows)} objets détectés.")

# ── Écriture Delta ────────────────────────────────────────────────────────────
df = spark.createDataFrame(rows)
df.display()

target_table = f"{catalog}.{registry_schema}.sql_object_registry"

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Table écrite : {target_table}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Dédoublonnage : si même object_name + schema, on garde le premier fichier trouvé
df_deduped = df.dropDuplicates(["object_name", "schema"])

# Ajout de la colonne 'copies' : nombre d'occurrences du même object_name
w = Window.partitionBy("object_name")
df_deduped = df_deduped.withColumn("copies", F.count("*").over(w))

print(f"Avant dédoublonnage : {df.count()} lignes")
print(f"Après dédoublonnage : {df_deduped.count()} lignes")

# Collisions réelles (même nom, schemas différents)
collisions_df = (
    df_deduped.filter(F.col("copies") > 1)
    .select("object_name", "copies", "schema", "object_type")
    .orderBy(F.col("copies").desc(), "object_name")
    .dropDuplicates(["object_name"])
)

print(f"\n{collisions_df.count()} noms en collision réelle (même nom, schemas différents) :")
collisions_df.display()

# Écriture avec le df dédoublonné + colonne copies
target_table = f"{catalog}.{registry_schema}.sql_object_registry"

(
    df_deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Table écrite : {target_table}")